In [1]:
# import dependencies
import pandas as pd
import numpy as np
import re
from rapidfuzz import process

In [2]:
# load acs data (for merging town names, GEOID, population and housing unit data)
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")

In [3]:
# open vt nfip claims data
claims = pd.read_csv("../data/raw/NFIP/vt_nfip_claims.csv")
claims.head()

,agricultureStructureIndicator,asOfDate,basementEnclosureCrawlspaceType,policyCount,crsClassificationCode,dateOfLoss,elevatedBuildingIndicator,elevationCertificateIndicator,elevationDifference,baseFloodElevation,...,rentalPropertyIndicator,state,reportedCity,reportedZipCode,countyCode,censusTract,censusBlockGroupFips,latitude,longitude,id
0,0,2026-02-11T00:00:00.000Z,2.0,1,NaN,2000-12-17T00:00:00.000Z,0,1,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,c97f77cf-d367-47bf-be38-1cee821b5629
1,0,2026-02-11T00:00:00.000Z,2.0,1,NaN,2011-05-26T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,085f146c-74c3-47e4-8b4c-60838f1faf0f
2,0,2026-02-11T00:00:00.000Z,1.0,1,NaN,1992-03-11T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5602.0,50023.0,5.002395e+10,5.002395e+11,44.3,-72.6,a48611f1-1e92-4e38-972c-ec42a556a8e1
3,0,2026-02-11T00:00:00.000Z,NaN,1,NaN,1981-02-19T00:00:00.000Z,0,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5476.0,50011.0,5.001101e+10,5.001101e+11,45.0,-72.7,65627989-987c-4916-bf1d-acc450ddd8e6
4,0,2026-02-11T00:00:00.000Z,NaN,1,NaN,2011-08-28T00:00:00.000Z,1,NaN,NaN,NaN,...,0,VT,Currently Unavailable,5748.0,50001.0,5.000196e+10,5.000196e+11,43.9,-72.9,d3eb3b79-d6de-473c-9939-535228b538d1


In [4]:
# clean nfipCommunityName to match town names in acs_df for eventual merging

manual_town_dict = {
    # spelling and punctuation errors
    "ENOSBURG TOWN": "ENOSBURGH TOWN",
    "FERRISBURG TOWN": "FERRISBURGH TOWN",
    "MORRISTOWNTOWN": "MORRISTOWN TOWN",
    "MT. HOLLY TOWN": "MOUNT HOLLY TOWN",
    "ST ALBANS TOWN": "ST. ALBANS TOWN",
    "HARDWICK TOWN AND TOWN": "HARDWICK TOWN",
    "NEWFANE TOWN AND TOWN": "NEWFANE TOWN",
    # villages inside of towns
    "MORRISVILLE TOWN": "MORRISTOWN TOWN",
    "JEFFERSONVILLE TOWN": "CAMBRIDGE TOWN",
    "ORLEANS TOWN": "BARTON TOWN",
}


# function to regex names
def clean_community(name):
    if pd.isna(name):
        return name
    name = name.upper()
    name = re.sub(r",", "", name)
    name = re.sub(r" OF", "", name)
    # Remove ' TOWN AND VILLAGE', ' VILLAGE', ' TOWN', ' CITY' at the end
    name = re.sub(r"VILLAGE", "TOWN", name)
    return name.strip()


# regex clean the names first, then apply manual corrections
claims["nfipCommunityName_clean"] = claims["nfipCommunityName"].apply(clean_community)
claims["nfipCommunityName_clean"] = claims["nfipCommunityName_clean"].replace(
    manual_town_dict
)
claims["nfipCommunityName_clean"].value_counts()

nfipCommunityName_clean
MONTPELIER CITY    208
BARRE CITY         206
WATERBURY TOWN     132
LUDLOW TOWN         68
RUTLAND CITY        54
                  ... 
ST. ALBANS CITY      1
STAMFORD TOWN        1
WINHALL TOWN         1
BARNARD TOWN         1
HINESBURG TOWN       1
Name: count, Length: 160, dtype: int64

In [5]:
# bin funding periods

# filter to match funding analysis period (FEMA HMA data starts in 1989)
claims = claims[claims["yearOfLoss"] >= 1989]


# bin years to match funding buckets
def claims_period(year):
    if pd.isna(year):
        return "unknown"
    if year < 2011:
        return "pre_2011_nfip_claims"
    elif year < 2023:
        return "2011_2022_nfip_claims"
    else:
        return "2023_plus_nfip_claims"


# apply function to create claims_period column
claims["claims_period"] = claims["yearOfLoss"].apply(claims_period)
claims["claims_period"].value_counts()

claims_period
2011_2022_nfip_claims    1724
2023_plus_nfip_claims     910
pre_2011_nfip_claims      691
Name: count, dtype: int64

In [6]:
# sum amount paid in insurance claims
claims["total_paid"] = (
    claims["amountPaidOnBuildingClaim"].fillna(0)
    + claims["amountPaidOnContentsClaim"].fillna(0)
    + claims["amountPaidOnIncreasedCostOfComplianceClaim"].fillna(0)
)

# aggregate count and dollar amount of claims by town name
claims_summary = (
    claims.groupby("nfipCommunityName_clean")
    .agg(nfip_claims=("id", "count"), total_nfip_claims_paid=("total_paid", "sum"))
    .reset_index()
)
claims_summary.head()

,nfipCommunityName_clean,nfip_claims,total_nfip_claims_paid
0,ANDOVER TOWN,3,38465.70
1,ARLINGTON TOWN,5,44870.80
2,BARNARD TOWN,1,0.00
3,BARNET TOWN,4,103197.02
4,BARRE CITY,206,9554298.45


In [7]:
# bin amount paid and count of claims by funding period

# pivot to get total claims paid per period as columns
claims_by_period = (
    claims.pivot_table(
        index="nfipCommunityName_clean",
        columns="claims_period",
        values="total_paid",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# pivot for count of claims per period
num_claims_by_period = (
    claims.pivot_table(
        index="nfipCommunityName_clean",
        columns="claims_period",
        values="id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# merge these back into claims_summary
claims_summary = claims_summary.merge(
    claims_by_period, on="nfipCommunityName_clean", how="left"
)
claims_summary = claims_summary.merge(
    num_claims_by_period,
    on="nfipCommunityName_clean",
    how="left",
    suffixes=("", "_count"),
)
claims_summary.shape

(160, 9)

In [8]:
# merge aggregated funding data back to town-level acs data on population and GEOID, to get a complete town-level dataset for analysis

# create town_name_upper in acs_df for merging
acs_df["town_name_upper"] = acs_df["town_name"].str.upper().str.strip()

# check for unmatched town names in claims_summary that are not in acs_df
unmatched = claims_summary[
    ~claims_summary["nfipCommunityName_clean"].isin(acs_df["town_name_upper"])
]
print(
    f'Unmatched towns (should be empty if successful):\n{unmatched["nfipCommunityName_clean"]}'
)

# outer join to keep all towns, even those without claims
nfip_df = pd.merge(
    acs_df[
        [
            "GEOID",
            "town_name",
            "town_name_upper",
            "total_population",
            "occupied_housing_units",
        ]
    ],
    claims_summary,
    left_on="town_name_upper",
    right_on="nfipCommunityName_clean",
    how="outer",
)

# fill remaining NaN values with 0
numeric_cols = nfip_df.select_dtypes(include=[np.number]).columns
nfip_df[numeric_cols] = nfip_df[numeric_cols].fillna(0)

# drop helper columns
nfip_df = nfip_df.drop(columns=["town_name_upper", "nfipCommunityName_clean"])

# checking for 256 towns
nfip_df.shape

Unmatched towns (should be empty if successful):
Series([], Name: nfipCommunityName_clean, dtype: object)


(256, 12)

In [9]:
# calculate claims per capita and claims per housing unit
nfip_df["claims_per_capita"] = np.where(
    nfip_df["total_population"] > 0,
    nfip_df["total_nfip_claims_paid"] / nfip_df["total_population"],
    np.nan,
)

nfip_df["claims_per_housing_unit"] = np.where(
    nfip_df["occupied_housing_units"] > 0,
    nfip_df["total_nfip_claims_paid"] / nfip_df["occupied_housing_units"],
    np.nan,
)

In [10]:
# display and export to csv
display(nfip_df.head())
nfip_df.to_csv("../data/cleaned/nfip_summary.csv", index=False)

,GEOID,town_name,total_population,occupied_housing_units,nfip_claims,total_nfip_claims_paid,2011_2022_nfip_claims,2023_plus_nfip_claims,pre_2011_nfip_claims,2011_2022_nfip_claims_count,2023_plus_nfip_claims_count,pre_2011_nfip_claims_count,claims_per_capita,claims_per_housing_unit
0,5000100325,Addison town,1175,490,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000
1,5001900475,Albany town,934,394,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000
2,5001300860,Alburgh town,1832,786,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000
3,5002701300,Andover town,616,238,3.0,38465.7,9285.32,0.0,29180.38,1.0,1.0,1.0,62.444318,161.620588
4,5000301450,Arlington town,2598,864,5.0,44870.8,44870.80,0.0,0.00,5.0,0.0,0.0,17.271286,51.933796
